# EasyAgent Cache Hit Rate Demo

这个 notebook 用来观察 **每次 `agent.invoke(...)` 之后的 cache 命中情况**。

它会展示两种场景：

1. `isolated`：每次调用前清空 history，只观察稳定前缀是否被 provider cache 命中。
2. `conversation`：保留 history，观察真实多轮对话里 cache 命中如何变化。

说明：

- 每次 `agent.invoke(...)` 内部可能对应 1 次或多次 LLM request，这里会把本次 invoke 期间新增的 LLM requests 聚合起来。
- `cache_hit_ratio = cache_hit_tokens / input_tokens`。
- 优先使用 `cacheReadTokens`；如果 provider 只返回 `cachedInputTokens`，则回退到它。
- 如果两者都没有，说明当前 endpoint **没有暴露真实 cache usage**，这时 notebook 仍会展示 `lastCacheBreak` 和 signature 稳定性，但命中率会是 `None`。

In [1]:
import os
import sys
import json
from pprint import pprint

current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from core.llm import EasyLLM
from agent.BasicAgent import BasicAgent


In [ ]:
# 选择 provider。
# 如果你想看 Anthropic-style 显式 cache marker，请用 anthropic_native。
# 如果你只想看 OpenAI-compatible usage，也可以切到 openai。

PROVIDER = "anthropic_native"  # "anthropic_native" or "openai"

if PROVIDER == "anthropic_native":
    llm = EasyLLM(
        provider="anthropic_native",
        base_url="http://127.0.0.1:5124",
        api_key="122",
        model="qwen3.5-9b",
    )
else:
    llm = EasyLLM(
        provider="openai",
        base_url="http://127.0.0.1:5124/v1",
        api_key="122",
        model="qwen3.5-9b",
    )

STABLE_SYSTEM_PROMPT = """
你是一个严谨的中文技术助手。

请始终遵守以下稳定规则：
1. 先给结论，再给理由。
2. 不要输出无关背景。
3. 如果信息不足，要明确说明假设。
4. 输出尽量结构化，但避免冗长。
5. 代码相关回答优先指出行为、约束和风险。
""".strip()

agent = BasicAgent(
    name="cache-hit-demo",
    llm=llm,
    system_prompt=STABLE_SYSTEM_PROMPT,
    reasoning={"effort": "low"},
    enable_tool=False,
)

print("provider:", llm.provider_name)
print("model:", llm.model)


provider: anthropic_native
model: qwen3.5-9b


In [3]:
def _all_llm_requests(agent):
    return list(agent.observability_recorder.export_state().get("llmRequests") or [])


def _cache_hit_tokens_for_request(item):
    if item.get("cacheReadTokens") is not None:
        return int(item.get("cacheReadTokens") or 0)
    if item.get("cachedInputTokens") is not None:
        return int(item.get("cachedInputTokens") or 0)
    return None


def _aggregate_invoke_requests(requests):
    input_tokens = sum(int(item.get("inputTokens") or 0) for item in requests)
    output_tokens = sum(int(item.get("outputTokens") or 0) for item in requests)
    total_tokens = sum(int(item.get("totalTokens") or 0) for item in requests)
    cache_read_tokens = sum(int(item.get("cacheReadTokens") or 0) for item in requests if item.get("cacheReadTokens") is not None)
    cached_input_tokens = sum(int(item.get("cachedInputTokens") or 0) for item in requests if item.get("cachedInputTokens") is not None)
    cache_hit_candidates = [_cache_hit_tokens_for_request(item) for item in requests]
    cache_hit_candidates = [item for item in cache_hit_candidates if item is not None]
    cache_hit_tokens = sum(cache_hit_candidates) if cache_hit_candidates else None
    cache_hit_ratio = (cache_hit_tokens / input_tokens) if (cache_hit_tokens is not None and input_tokens > 0) else None
    return {
        "llm_requests": len(requests),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "cache_read_tokens": cache_read_tokens if cache_read_tokens > 0 else None,
        "cached_input_tokens": cached_input_tokens if cached_input_tokens > 0 else None,
        "cache_hit_tokens": cache_hit_tokens,
        "cache_hit_ratio": cache_hit_ratio,
    }


def invoke_and_measure(agent, query, *, reset_history=False, label=None):
    if reset_history:
        agent.clear_history()

    before = _all_llm_requests(agent)
    before_count = len(before)
    result = agent.invoke(query)
    after = _all_llm_requests(agent)
    new_requests = after[before_count:]
    summary = _aggregate_invoke_requests(new_requests)
    obs_summary = agent.observability_recorder.get_summary()
    context_usage = agent.get_context_usage()
    return {
        "label": label or query[:40],
        "query": query,
        "response": result,
        **summary,
        "cache_breaks_total": obs_summary.get("cacheBreaks"),
        "last_cache_break": obs_summary.get("lastCacheBreak"),
        "cache_state": context_usage.get("cache"),
    }


def print_turn_report(row):
    print("=" * 80)
    print("label:", row["label"])
    print("query:", row["query"])
    print("response:", row["response"])
    print("llm_requests:", row["llm_requests"])
    print("input_tokens:", row["input_tokens"])
    print("output_tokens:", row["output_tokens"])
    print("total_tokens:", row["total_tokens"])
    print("cache_read_tokens:", row["cache_read_tokens"])
    print("cached_input_tokens:", row["cached_input_tokens"])
    print("cache_hit_tokens:", row["cache_hit_tokens"])
    print("cache_hit_ratio:", row["cache_hit_ratio"])
    print("last_cache_break:")
    pprint(row["last_cache_break"])
    print("cache_state:")
    pprint(row["cache_state"])


## 1. Isolated 模式：每次调用前清空 history

这个模式最容易看 provider prompt cache 是否生效。

预期：

- 第 1 次一般是 warm-up，`cache_hit_ratio` 常常接近 `0` 或 `None`。
- 第 2、3 次如果 endpoint 支持 cache usage，命中率应该上升。
- 如果一直是 `None`，说明 endpoint 没有返回 cache usage。

In [5]:
isolated_results = []
from core import enable_logging
enable_logging()
for i in range(3):
    row = invoke_and_measure(
        agent,
        "请用两句话解释为什么稳定的 system prompt 和稳定的 tool schema 有助于 prompt cache 命中。",
        reset_history=True,
        label=f"isolated_turn_{i+1}",
    )
    isolated_results.append(row)
    print_turn_report(row)


2026-04-30 17:44:33,016 | INFO | 对话历史已清空
2026-04-30 17:44:33,016 | INFO | 使用普通模式调用智能体
2026-04-30 17:44:35,441 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"
2026-04-30 17:44:35,451 | INFO | 对话历史已清空
2026-04-30 17:44:35,452 | INFO | 使用普通模式调用智能体


label: isolated_turn_1
query: 请用两句话解释为什么稳定的 system prompt 和稳定的 tool schema 有助于 prompt cache 命中。
response: 

结论：稳定的 system prompt 和 tool schema 确保每次请求的输入结构一致，使缓存匹配算法能准确识别历史请求。

理由：Prompt cache 基于内容哈希匹配，稳定的 prompt 和 schema 减少输入内容波动，显著提升缓存命中率，降低重复计算开销。
llm_requests: 1
input_tokens: 1456
output_tokens: 180
total_tokens: 1636
cache_read_tokens: None
cached_input_tokens: None
cache_hit_tokens: None
cache_hit_ratio: None
last_cache_break:
None
cache_state:
{'anchorActive': True,
 'anchorProvider': 'anthropic_native',
 'anchorReplayIndex': 2,
 'enabled': True,
 'lastBreak': None,
 'lastCacheUsage': {'cacheReadTokensForBreakDetection': None,
                    'inputTokens': 1456,
                    'outputTokens': 180,
                    'totalTokens': 1636,
                    'usageSource': 'provider'},
 'lastSignature': {'cache_policy_hash': 'ccf9424ec1724009eb0e9478c7c05434459f4651b8c1d4985ae28758d3da8b6e',
                   'extra_hash': '74234e98afe7498fb5daf1f36ac2d78acc339464f95070

2026-04-30 17:45:23,002 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"


LLMInvokeError: LLM 返回了空响应!

## 2. Conversation 模式：保留 history

这个模式更接近真实 agent 场景。

这里你会看到：

- history 在增长；
- 但稳定的 system/tools prefix 仍然应该可复用；
- 如果后面切 reasoning 或改 system prompt，`lastCacheBreak` 会告诉你为什么 cache 失效。

In [ ]:
agent.clear_history()
conversation_queries = [
    "先用一句话解释 prompt cache 的核心思想。",
    "在上一句基础上，再补一句说明为什么动态 memory 不应该进入稳定前缀。",
    "继续补一句，说明为什么 runtime skill context 只应该作为动态 replay context。",
]

conversation_results = []
for i, query in enumerate(conversation_queries, start=1):
    row = invoke_and_measure(agent, query, reset_history=False, label=f"conversation_turn_{i}")
    conversation_results.append(row)
    print_turn_report(row)


## 3. 人为制造一次 cache break

这里把 `reasoning` 从 `low` 改成 `high`，然后再调用一次。

预期：

- `lastCacheBreak.reason` 可能出现 `cache_signature_changed`。
- `changedFields` 里通常会包含 `reasoning_hash`。

In [ ]:
agent.reasoning = {"effort": "high"}
cache_break_row = invoke_and_measure(
    agent,
    "继续一句话说明，切换 reasoning 配置为什么会导致 cache signature 变化。",
    reset_history=False,
    label="reasoning_changed_turn",
)
print_turn_report(cache_break_row)
